# A3c -- the somatic vs non-somatic DE baseline

Answers **Reviewer #2's other round-1 suggestion**, offered as an alternative to the pseudo-granule
control and likewise never run:

> "As a potential control, the authors could consider performing a **differential expression
> analysis between somatic RNA and all non-somatic RNA, independent of granule detection**, and
> then assess to what extent the observed granule-specific differences **exceed or diverge from**
> this baseline non-somatic signal."

The framing sentence just before it is what makes this two-axis rather than one:

> "This concern applies not only to the reported granule enrichments, but also to ... the
> comparison of granule compositions between WT and AD conditions ... it is difficult to exclude
> the possibility that some of the reported differences **between regions or conditions** are
> driven by systematic differences in ambient RNA levels."

### What already exists, and what it is missing

`code/benchmark/benchmark_ambient.ipynb` does **not** answer this. It is an OLS regression of
per-spot granule *density* on `AD + ambient_marker_cov` -- no gene-level contrast, no somatic
layer, and its `ambient = extrasomatic - granule_expression` is defined by subtracting called
granules, so it is not independent of detection. That is the analysis round 2 dismissed.

`code/old/benchmark_diffusion.ipynb` **is** Axis 1, already implemented -- `baseline_logFC`,
`granule_enrichment`, `delta`, and a non-marker regression -- with an R panel at
`code/figures_response.Rmd:1421-1452` under a heading literally titled *"Reviewer 2, Major Comment
9"*. It was built for round 1 and then not used; its output CSV is no longer on disk. This notebook
revives it and closes six gaps:

| # | gap in `benchmark_diffusion.ipynb` | fixed in |
|---|---|---|
| 1 | WT only -- the reviewer names *conditions* | §5 |
| 2 | `delta` subtracts two logFCs with different references (`USE_ALT_GRANULE_VS_SOMA` ships **off**) | §3 |
| 3 | the baseline includes in-granule transcripts, so it contains the signal it is a null for | §1 |
| 4 | no significance on `granule_enrichment` or `delta` | §4 |
| 5 | uses `all_granules` + post-hoc filtering, and keys `nc_ratio` on `sphere_z` where `nc_filter` uses `layer_z` | §1 |
| 6 | cell 7 is O(n_spots x n_transcripts) -- a 103M-element mask per spot | §1 |

### Where this sits in the run

Needs nothing from the HGCC array, so it can run while the array is still queued -- and its section 1 caches the transcript partition, which is reused within this notebook only (A3a no longer reads it). Runs **once, top to bottom, with nothing to adjust** -- see the runbook in
`README.md`.

**Run this notebook from `R2_revision/ambient_controls/`, on the `mcDETECT-env` kernel.**

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, wilcoxon

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- runtime gates -------------------- #
# THE DEFAULTS BELOW PRODUCE THE FINAL TABLES. Run top to bottom, once, and change nothing.
# There is no DRY_RUN here: this notebook is transcript-level throughout, so there is no sphere
# count to subsample -- section 1's partition is the cost, and it is cached.
OVERWRITE = False        # True -> recompute the cached transcript partition
VALIDATE = True          # section 7 correctness gates. On by default (A1/A2 have them off): they
                         #   are the LAST section, so every table is already written before they
                         #   run, and the partition-completeness gate is what proves the three
                         #   layers really are disjoint.
RUN_CLIP_BIAS = True     # section 2 -- quantifies the published subtraction's bias
RUN_COUNT_MODEL = True   # section 4 -- quasi-Poisson on raw counts (the primary inference)
RUN_NONSEED = True       # section 5 -- THE non-circular test: genes that seeded nothing
RUN_AXIS2 = True         # section 6 -- WT/AD on the three layers

C.ensure_dirs()
OUT = C.A3C_DIR

print("writing to:", OUT)
print("layers    :", C.DE_LAYERS)
print("primary   :", C.DE_COUNT_MODEL, "| secondary:", C.DE_PUBLISHED_METHOD)
for r in C.REPORTING_RULES:
    print("\nRULE:", r)

writing to: /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/a3c
layers    : ['intrasomatic', 'granule', 'residual_extrasomatic']
primary   : quasipoisson | secondary: wilcoxon

RULE: Significant-gene COUNTS are not comparable across layers: the layers differ in counts per spot and in sparsity, and a rank test's power tracks that -- which is why ambient/cell show 253/234 significant genes vs granule's 161. Compare RANKINGS and logFC correlations only. Quoting the tallies invites the reading 'the authors' ambient layer yields more DE genes than their granule layer'.

RULE: n = 1 vs 1. One WT section, one AD section, so every spot-level WT/AD p-value is pseudo-replication. Frame WT/AD descriptively and put the inferential weight on the granule-vs-residual-ambient DIVERGENCE, which is a within-sample comparison.


/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

## 1. The partition -- built at transcript level

Three **disjoint** arms, assigned per transcript by one batched ball query over every Set-2 sphere:

| arm | definition |
|---|---|
| `intrasomatic` | `overlaps_nucleus == 1` |
| `granule` | inside some Set-2 sphere **and** `overlaps_nucleus == 0` |
| `residual_extrasomatic` | neither |

The partition supports **both** baselines §3 needs, from the same three counts:

* **all non-somatic RNA** = `granule + residual_extrasomatic`. This is the reviewer's literal
  wording and it is genuinely **independent of granule detection** -- neither layer needs a sphere
  to be defined. It is the primary baseline.
* **residual extrasomatic alone**, which excludes the in-granule transcripts so the baseline does
  not contain the signal it is a null for. Note what this costs: `residual_extrasomatic` means
  "extrasomatic **and not inside a called sphere**", so it is detection-**dependent** by
  construction and must never be described otherwise. It is reported as a sensitivity arm.

Including the granule transcripts biases the difference **toward zero**, so the primary is also
the more conservative of the two -- there is no tension between "literal" and "safe" here. The
gate in §6 asserts the three arms sum to the transcript count exactly, per gene, per sample.

It also sidesteps the spot-matrix subtraction entirely -- see §2 for why that matters.

In [2]:
part_path = OUT / "partition_counts.csv"

if part_path.exists() and not OVERWRITE:
    parts = pd.read_csv(part_path)
    print(f"loaded cached partition: {parts.shape}")
else:
    rows = []
    for sample in C.SAMPLES:
        tx = A3.load_transcripts(sample)
        granules = pd.read_parquet(C.mcdetect_granules_path(sample))
        # Cached to C.transcript_layer_path(sample), and used only within this notebook -- A3a
        # does NOT read it (the analysis that did, the adaptive-threshold re-test, was removed).
        # OVERWRITE must be threaded into `cache`: the parquet is keyed on the sample alone, so
        # without this it is reused on existence and a changed granule set, buffer or z_col is
        # silently ignored. DE_SPHERE_BUFFER changed, so this matters.
        layer = A3.partition_transcripts(tx, granules, sample=sample, cache=not OVERWRITE)
        # `target` is categorical in these parquets, and crosstab emits a row for EVERY
        # category including unobserved ones -- that inflates the file and n_genes downstream.
        tgt = tx["target"]
        if isinstance(tgt.dtype, pd.CategoricalDtype):
            tgt = tgt.cat.remove_unused_categories()
        tab = (pd.crosstab(tgt, layer)
               .reindex(columns=C.DE_LAYERS, fill_value=0)
               .reset_index().rename(columns={"target": "gene"}))
        tab["sample"] = sample
        tab["n_total"] = tab[C.DE_LAYERS].sum(axis=1)
        rows.append(tab)
        print(f"[{sample}] " + " | ".join(
            f"{l}: {int(tab[l].sum()):,}" for l in C.DE_LAYERS))
        del tx
    parts = pd.concat(rows, ignore_index=True)
    parts.to_csv(part_path, index=False)

# `parts` stays COMPLETE -- it is the partition of EVERY transcript, and section 6's gate (a)
# checks exactly that exhaustiveness against the raw row count. partition_counts.csv is therefore
# a faithful record and section 3.1's layer totals are the true totals.
#
# `parts_panel` is what the ANALYSES use. The transcript tables carry 25 Blank probes alongside
# the 290 panel genes; left in, they reach the Axis-2 correlations (n = 315, not 290), the
# quasi-Poisson BH family (m = 630, not 580) and the figures, and they make the response document
# report 295 "non-marker genes" one paragraph after it says 270. Blanks carry no biological
# signal, so they are dropped once, here, and every downstream table inherits a clean panel.
_panel = set(A3.load_genes("WT"))
parts_panel = parts[parts["gene"].isin(_panel)].reset_index(drop=True)
print(f"panel filter: {len(parts)} rows -> {len(parts_panel)} "
      f"({len(parts) - len(parts_panel)} non-panel probe rows held out of the analyses)")

display(parts.groupby("sample")[C.DE_LAYERS + ["n_total"]].sum())

loaded cached partition: (630, 6)
panel filter: 630 rows -> 580 (50 non-panel probe rows held out of the analyses)


,intrasomatic,granule,residual_extrasomatic,n_total
sample,,,,
AD,20901924,3677422,44297301,68876647
WT,28838683,6128938,68430447,103398068


## 2. Why not the published spot-matrix subtraction

`7_neuropil_subdomains.ipynb` cell 9 and `benchmark_ambient.ipynb` cell 6 both build their ambient
layer as

```python
np.maximum(spots.layers["extrasomatic_transcripts"] - spot_granule_expression, 0)
```

which compounds three errors:

1. **Misallocation across tiles.** `spot_embedding` assigns each granule to the spot containing its
   **centre** (`downstream.py:706-712`), but the sphere spans neighbouring spots -- so granule
   counts are subtracted from the wrong tile near every boundary.
2. **Over-subtraction.** `profile()` counts **all** transcripts in the sphere, including
   `overlaps_nucleus == 1`. The in-soma filter caps *markers* at 10% and does not constrain
   non-markers at all, so intranuclear transcripts are subtracted from an extrasomatic-only layer.
3. **Double counting.** Overlapping granules both claim their shared transcripts -- and since the
   merge rule requires centres within `0.4*r`, granules routinely overlap without merging.

The `np.maximum(..., 0)` clip makes the resulting bias one-sided. **Measured, it is small and it
is NOT marker-biased**: non-markers go negative in a median 0.046% of spots (max 0.32%) against
0.000% (max 0.023%) for the 20 markers -- roughly 24x *less* exactly where the published result
lives. So the transcript-level rebuild is justified by the three structural errors above, and the
clip is reported as quantified-and-negligible rather than as a criticism of the published figure. The cell below
quantifies it: per gene, the fraction of spots where the raw difference is negative *before*
clipping. That number is the size of the floor being silently applied.

### Scope: this is measured on the published ROI, not the whole section

`neuropil_subdomains_spots_ambient.h5ad` is **4,310 spots covering Isocortex (3,931) + FT (379)**
at 50 um, both samples pooled -- not the whole section. That is deliberate: it is the object
`benchmark_ambient.ipynb` cell 6 actually wrote, with the same `spot_width = 50`,
`granule_subtype_kmeans`, `include_soma_features = False`, `smoothing = False` used below, and the
claim being quantified is a claim about *that* subtraction. `spot_embedding` assigns `-1` to any
granule outside every spot box and drops it (`valid_mask`), so the ~1.08M whole-section granules
are correctly restricted rather than clipped onto boundary spots. `clip_bias_scope.csv` records
the restriction so the per-gene table cannot later be quoted as whole-section.

The 25 um whole-section `neuropil_subdomains_spots.h5ad` also carries
`extrasomatic_transcripts` and could extend this to every region. Deliberately not run: the
published subtraction is the target, and a second grid would be a new claim to defend.

In [3]:
if RUN_CLIP_BIAS:
    from mcDETECT.downstream import spot_embedding

    spots = sc.read_h5ad(C.SUBDOMAIN_SPOTS_50)
    # COORDINATE FRAME. granule_adata_tsne.h5ad stores obs["global_x"] in each sample's OWN raw
    # frame (219..6967), while these spots are on the combined canvas (-14..12014). Pairing the
    # two assigns nearly every granule to the wrong spot or none, and the whole clip-bias table
    # becomes noise. neuropil_subdomains_granule_adata.h5ad is the same object already moved onto
    # the canvas by 5_neuropil_subdomains_data.py:203-206 (0..12532) -- and it already carries
    # granule_subtype_kmeans, so the label merge is unnecessary too.
    gad = sc.read_h5ad(C.SUBDOMAIN_GRANULE_ADATA)
    gad.obs["granule_subtype_kmeans"] = gad.obs["granule_subtype_kmeans"].astype("category")

    # FRAME, NOT COVERAGE. Do NOT rewrite this as "granule bbox inside spot bbox" -- that fails
    # for a correct pairing. The spots are an ROI (Isocortex + FT, y 2830..6146) while gad is the
    # whole section (y 75..6362), so the granule extent is LARGER by construction. What must hold
    # is that both sit on the same canvas, and the containment therefore runs the other way:
    # the spot ROI must lie inside the granule extent. Checked PER BATCH and on BOTH axes -- WT
    # and AD occupy disjoint x bands here (WT 0..5139, AD 7800..12532), so a per-batch check
    # catches a mis-registered sample that a pooled bbox hides, and the y axis is where a raw
    # per-sample frame (granule_adata_tsne.h5ad) diverges most.
    PAD = float(C.SPOT_GRID)
    for b, ssub in spots.obs.groupby("batch", observed=True):
        gsub = gad.obs[gad.obs["batch"] == b]
        assert len(gsub), f"no granules for batch {b} -- batch labels do not match"
        for ax in ("global_x", "global_y"):
            lo, hi = float(ssub[ax].min()), float(ssub[ax].max())
            glo, ghi = float(gsub[ax].min()), float(gsub[ax].max())
            assert glo - PAD <= lo and hi <= ghi + PAD, (
                f"{b} {ax}: spot ROI {lo:.0f}..{hi:.0f} is not inside the granule extent "
                f"{glo:.0f}..{ghi:.0f} -- wrong coordinate frame")
        print(f"[ok] {b}: spot ROI inside the granule extent on both axes")
    assert list(gad.var_names) == list(spots.var_names), (
        "gene axes differ; spot_granule_expression is returned on granule_adata.var_names order "
        "and would be silently mislabelled by the subtraction below")

    # Granules whose centre lies in the ROI bounding box. The ROI is not a full rectangle, so
    # this is an upper bound on what can be assigned -- it is the denominator for the coverage
    # check after the call, which is the thing a bbox comparison cannot tell us.
    sxb = (spots.obs["global_x"].min() - PAD, spots.obs["global_x"].max() + PAD)
    syb = (spots.obs["global_y"].min() - PAD, spots.obs["global_y"].max() + PAD)
    in_bbox = ((gad.obs["global_x"] >= sxb[0]) & (gad.obs["global_x"] <= sxb[1]) &
               (gad.obs["global_y"] >= syb[0]) & (gad.obs["global_y"] <= syb[1]))
    n_in_bbox = int(in_bbox.sum())
    print(f"ROI: {spots.n_obs:,} spots ({dict(spots.obs['brain_area'].value_counts())}) | "
          f"granules in ROI bbox {n_in_bbox:,}/{gad.n_obs:,}")

    _, _, aux, spot_gnl, _ = spot_embedding(
        spots=spots, granule_adata=gad,
        spot_loc_key=("global_x", "global_y"), spot_width=C.SPOT_GRID,
        spot_height=C.SPOT_GRID, granule_loc_key=("global_x", "global_y"),
        granule_subtype_key="granule_subtype_kmeans",
        subtype_names=[str(i) for i in
                       range(gad.obs["granule_subtype_kmeans"].nunique())],
        granule_count_layer="counts", include_soma_features=False, smoothing=False)

    # The check the bbox gate cannot give: granules must actually land IN spots. spot_embedding
    # drops the unassigned (assigned_spot = -1, valid_mask), so a frame error does not raise --
    # it silently builds the table below on almost nothing. The ROI's spots tile ~10.8 mm2 of a
    # ~39.9 mm2 bounding box, so ~25-30% is the expected rate; the floor only catches collapse.
    n_assigned = int(np.sum(aux["granule_count"]))
    frac_assigned = n_assigned / max(n_in_bbox, 1)
    assert n_assigned > 0, "no granule was assigned to any spot -- wrong coordinate frame"
    assert frac_assigned > 0.10, (
        f"only {frac_assigned:.1%} of ROI-bbox granules landed in a spot -- suspect the frame")
    print(f"assigned {n_assigned:,} granules to spots ({frac_assigned:.1%} of the ROI bbox)")

    extra = np.asarray(spots.layers["extrasomatic_transcripts"], dtype=float)
    raw_diff = extra - np.asarray(spot_gnl, dtype=float)
    neg = raw_diff < 0

    clip = pd.DataFrame({
        "gene": list(spots.var_names),
        "frac_spots_negative": neg.mean(axis=0),
        "mean_negative_mass": np.where(neg, -raw_diff, 0).mean(axis=0),
        "is_marker": [g in set(C.SYN_GENES) for g in spots.var_names],
    }).sort_values("frac_spots_negative", ascending=False)
    clip.to_csv(OUT / "clip_bias_by_gene.csv", index=False)

    # The restriction travels with the numbers, so clip_bias_by_gene.csv can never be quoted as
    # a whole-section result.
    pd.DataFrame([dict(
        spots_object=C.SUBDOMAIN_SPOTS_50.name, grid_um=C.SPOT_GRID, smoothing=False,
        n_spots=int(spots.n_obs),
        brain_areas=";".join(f"{k}:{v}" for k, v in
                             spots.obs["brain_area"].value_counts().items()),
        samples=";".join(map(str, spots.obs["batch"].unique())),
        n_granules_total=int(gad.n_obs), n_granules_in_roi_bbox=n_in_bbox,
        n_granules_assigned=n_assigned, frac_assigned_of_bbox=frac_assigned,
        n_genes=int(spots.n_vars),
    )]).to_csv(OUT / "clip_bias_scope.csv", index=False)

    print(f"overall: {neg.mean():.4f} of gene x spot cells are negative before clipping")
    print(clip.groupby("is_marker")["frac_spots_negative"]
          .agg(["mean", "median", "max"]).to_string())
    display(clip.head(20))

[ok] MERSCOPE_WT_1: spot ROI inside the granule extent on both axes
[ok] MERSCOPE_AD_1: spot ROI inside the granule extent on both axes
ROI: 4,310 spots ({'Isocortex': 3931, 'FT': 379}) | granules in ROI bbox 683,708/1,080,146


assigned 311,000 granules to spots (45.5% of the ROI bbox)
overall: 0.0005 of gene x spot cells are negative before clipping
               mean    median       max
is_marker                              
False      0.000551  0.000464  0.003248
True       0.000023  0.000000  0.000232


,gene,frac_spots_negative,mean_negative_mass,is_marker
122,Htr1f,0.003248,0.003248,False
268,Ptprc,0.003016,0.004176,False
210,Apod,0.002552,0.002784,False
60,Dcx,0.002552,0.002552,False
64,Aldh1a2,0.002320,0.002320,False
251,Cbln1,0.002320,0.002552,False
186,Nell1,0.002088,0.002088,False
156,Hfe,0.002088,0.002320,False
255,Aif1,0.002088,0.002088,False
0,Igf2,0.002088,0.002088,False


## 3. Axis 1 -- compartment

The reviewer's quantity, computed against **both** candidate baselines with all three references
made consistent:

```
baseline_all_logFC = log2 share(granule + residual_extrasomatic) - log2 share(intrasomatic)
baseline_logFC     = log2 share(residual_extrasomatic)           - log2 share(intrasomatic)
granule_enrichment = log2 share(granule)                         - log2 share(intrasomatic)
delta_all          = granule_enrichment - baseline_all_logFC     # PRIMARY
delta              = granule_enrichment - baseline_logFC         # sensitivity
```

**`baseline_all_logFC` is primary.** The reviewer asked for DE "between somatic RNA and all
non-somatic RNA, *independent of granule detection*", and `granule + residual_extrasomatic` is
exactly that -- no sphere is needed to define either side. `residual_extrasomatic` alone is
granule-free, which is desirable, but it is defined as "extrasomatic **and not inside a called
sphere**" and is therefore detection-**dependent**; it is kept as a labelled sensitivity arm, never
presented as the detection-independent baseline. Including the granule transcripts biases the
difference toward zero, so the primary is the harder test as well as the literal one.

### Why soma is the reference, when granules are extrasomatic

It is a presentation device, not a claim that soma is the biologically apt comparator. A shared
denominator puts the two quantities on one axis so they can be plotted and regressed against each
other -- and it then **cancels exactly** out of the difference:

```
delta = [log2 sh_gnl - log2 sh_soma] - [log2 sh_res - log2 sh_soma] = log2 sh_gnl - log2 sh_res
```

So `delta` *is* granule-versus-extrasomatic and carries no soma term at all; §6 asserts this to
1e-9. `benchmark_diffusion.ipynb` used a **different** reference on each axis, so its `delta`
subtracted two logFCs sharing no denominator -- its `USE_ALT_GRANULE_VS_SOMA` branch is the correct
form and ships switched **off**. Here it is the primary.

The cancellation does **not** extend to `residual`: the fitted slope is ~1.18, not 1, so that
statistic does depend on the soma layer. Both statistics are tested in §4 and they agree.

Then a regression fitted on **non-markers** gives the reference line; markers above it are enriched
beyond what the baseline predicts.

**Frame the claim as divergence, not excess.** The reviewer's own wording is "exceed **or diverge
from**", and divergence is the safer and stronger of the two: `normalize_total`-style compositional
scaling makes an absolute `|logFC|` comparison fragile, whereas a rank correlation and the residual
from the non-marker line are not.


In [4]:
axis1_frames, axis1_stats = [], []

for sample in C.SAMPLES:
    p = parts_panel[parts_panel["sample"] == sample].set_index("gene")
    genes = [g for g in A3.load_genes(sample) if g in p.index]
    counts_by_layer = {l: p[l].to_dict() for l in C.DE_LAYERS}

    df, reg = A3.axis1_table(counts_by_layer, genes, markers=C.SYN_GENES)
    df["sample"] = sample
    axis1_frames.append(df)

    m = df["is_marker"]
    row = dict(sample=sample, n_genes=len(df), n_markers=int(m.sum()))
    # `_all` is the PRIMARY (reviewer's literal baseline); "" is the granule-free sensitivity.
    for suffix, xcol in [("_all", "baseline_all_logFC"), ("", "baseline_logFC")]:
        rho, pv = spearmanr(df[xcol], df["granule_enrichment"])
        row.update({
            f"spearman_rho{suffix}": float(rho),
            f"spearman_p{suffix}": float(pv),
            f"reg_slope{suffix}": reg[f"slope{suffix}"],
            f"reg_intercept{suffix}": reg[f"intercept{suffix}"],
            f"markers_above_diagonal{suffix}": int((df.loc[m, "granule_enrichment"]
                                                    > df.loc[m, xcol]).sum()),
            f"markers_above_regression{suffix}": int(df.loc[m, f"above_regression{suffix}"].sum()),
            f"median_delta{suffix}_marker": float(df.loc[m, f"delta{suffix}"].median()),
            f"median_delta{suffix}_other": float(df.loc[~m, f"delta{suffix}"].median()),
            f"median_residual{suffix}_marker": float(df.loc[m, f"residual{suffix}"].median()),
            f"median_residual{suffix}_other": float(df.loc[~m, f"residual{suffix}"].median()),
        })
    axis1_stats.append(row)

axis1 = pd.concat(axis1_frames, ignore_index=True)
axis1.to_csv(OUT / "axis1_gene_table.csv", index=False)
stats1 = pd.DataFrame(axis1_stats)
stats1.to_csv(OUT / "axis1_summary.csv", index=False)
display(stats1[["sample", "n_genes", "spearman_rho_all", "reg_slope_all",
                "markers_above_regression_all", "median_delta_all_marker",
                "median_delta_all_other", "markers_above_regression"]])


,sample,n_genes,spearman_rho_all,reg_slope_all,markers_above_regression_all,median_delta_all_marker,median_delta_all_other,markers_above_regression
0,WT,290,0.814725,1.197814,18,0.228219,-0.601747,19
1,AD,290,0.819813,1.147811,18,0.082622,-0.881552,18


## 4. Axis 1 -- significance, and a non-compositional primary

Two problems with testing this the published way, both handled here.

**Compositionality.** `sc.pp.normalize_total(target_sum=1e4)` makes every comparison a comparison of
*shares*. The 20 markers are ~31% of all transcripts (WT 0.3235 / AD 0.2960), so a real marker
enrichment mechanically depletes every other gene, and the baseline inherits the mirror image. The
primary inference is therefore a **count model on raw counts with a `log(layer total)` offset**
(quasi-Poisson), which has no such coupling. The Wilcoxon path is kept as a secondary arm so the
numbers stay comparable to the published CSVs.

**Nothing was ever tested.** `benchmark_diffusion.ipynb` gives the baseline a paired spot-level
Wilcoxon and gives `granule_enrichment` and `delta` nothing at all. Below, `delta` is tested two
ways: a spot-level paired Wilcoxon per gene, and a one-sided marker-set enrichment on the
**residual** ranking -- which is the divergence claim stated as a test.

In [5]:
if RUN_COUNT_MODEL:
    import statsmodels.api as sm

    # WHY PER-SPOT. The obvious form -- two aggregate numbers per gene (granule total, residual
    # total) with a log(layer total) offset -- is SATURATED: 2 observations, 2 parameters,
    # df_resid = 0. statsmodels does not raise; it returns scale = inf, bse = inf and pval
    # EXACTLY 1.0 for every gene, so the table looks like real output and is not. The point
    # estimate is fine (it is algebraically the composition logFC); only the inference is dead.
    #
    # Aggregating to spots gives 2 x n_spots observations per gene, real residual df, and a
    # genuine over-dispersion estimate -- which is the whole reason for preferring a count model
    # over the compositional Wilcoxon in the first place.
    spot_path = OUT / "spot_layer_counts.parquet"
    if spot_path.exists() and not OVERWRITE:
        spot_counts = pd.read_parquet(spot_path)
    else:
        rows = []
        for sample in C.SAMPLES:
            tx = A3.load_transcripts(sample)
            layer = A3.partition_transcripts(tx, pd.read_parquet(
                C.mcdetect_granules_path(sample)), sample=sample)
            spots = sc.read_h5ad(C.spots_path(sample))
            sf = A3._import_sphere_features()
            # spot id per transcript, on the published grid
            gl = C.SPOT_GRID
            sx = spots.obs["global_x"].to_numpy(); sy = spots.obs["global_y"].to_numpy()
            ix = np.round((tx["global_x"].to_numpy() - sx.min()) / gl).astype(np.int64)
            iy = np.round((tx["global_y"].to_numpy() - sy.min()) / gl).astype(np.int64)
            spot_id = ix * (iy.max() + 1) + iy
            tgt = tx["target"]
            if isinstance(tgt.dtype, pd.CategoricalDtype):
                tgt = tgt.cat.remove_unused_categories()
            agg = (pd.DataFrame({"spot": spot_id, "gene": tgt.to_numpy(),
                                 "layer": layer.to_numpy()})
                   .groupby(["spot", "gene", "layer"], observed=True).size()
                   .rename("n").reset_index())
            agg["sample"] = sample
            rows.append(agg)
            del tx
        spot_counts = pd.concat(rows, ignore_index=True)
        A3.write_parquet_atomic(spot_counts, spot_path)
    print(f"per-spot layer counts: {len(spot_counts):,} rows")

    # TWO FITS, DIFFERENT GENE POOLS.
    #
    # (a) PANEL pool -- all 290 panel genes in the offset. This is the marker-vs-rest comparison
    #     reported in section 3.5, and the pool has to include the markers for that to mean
    #     anything.
    #
    # (b) NEUTRAL pool -- only the 252 genes that neither seeded detection nor filtered it, in
    #     BOTH the offset and the fitted set. Section 5 needs this one. With the markers in the
    #     denominator every neutral gene's granule share is deflated by a constant (the markers
    #     are ~31% of transcripts and sit inside granules by construction), so zero stops being
    #     the neutral point and the analysis degrades to a statement about relative ordering.
    #     Summed over the neutral genes alone, logFC = 0 means "the same share of granule RNA as
    #     of the surrounding RNA" -- which is exactly the null the reviewer's question implies.
    qp = A3.fit_layer_count_model(spot_counts, _panel)          # Blank probes carry no signal
    qp.to_csv(OUT / "axis1_count_model.csv", index=False)
    assert (qp["df_resid"].dropna() > 0).all(), "some fits were saturated"
    print(f"median over-dispersion phi = {qp['dispersion'].median():.2f} "
          f"(>> 1 means a Poisson cutoff would under-correct)")
    display(qp.groupby(["sample", "is_marker"])[["logFC_granule_vs_residual", "pval"]]
            .agg({"logFC_granule_vs_residual": "median", "pval": lambda x: (x < 0.05).mean()}))

    neutral = A3.neutral_genes()
    print(f"\nneutral pool: {len(neutral)} genes (no seed, no negative control)")
    qpn = A3.fit_layer_count_model(spot_counts, neutral)
    assert not qpn["is_marker"].any(), "a seed gene reached the neutral count model"
    qpn.to_csv(OUT / "axis1_count_model_neutral.csv", index=False)
    print("neutral-pool median logFC per sample:")
    display(qpn.groupby("sample")["logFC_granule_vs_residual"]
            .agg(median="median", frac_above_zero=lambda x: (x > 0).mean()))


per-spot layer counts: 9,978,031 rows


[WT] spots with non-zero exposure in both layers: 9,917/12,153


[AD] spots with non-zero exposure in both layers: 8,181/9,923


median over-dispersion phi = 3.93 (>> 1 means a Poisson cutoff would under-correct)


logFC_granule_vs_residual      pval
sample is_marker                                     
AD     False                      -0.929328  0.981481
       True                        0.085438  0.800000
WT     False                      -0.646067  0.944444
       True                        0.239287  0.950000


neutral pool: 252 genes (no seed, no negative control)


[WT] spots with non-zero exposure in both layers: 9,806/11,855


[AD] spots with non-zero exposure in both layers: 7,959/9,899


neutral-pool median logFC per sample:


,median,frac_above_zero
sample,,
AD,-0.047366,0.460317
WT,0.153036,0.607143


In [6]:
# Divergence stated as a test: are the granule markers enriched in the POSITIVE residual from
# the non-marker regression line? One-sided Mann-Whitney, which needs no distributional assumption
# and is invariant to the compositional rescaling that makes |logFC| fragile.
#
# Run against BOTH baselines. `delta_all` / `residual_all` use the reviewer's literal
# "all non-somatic RNA" reference and are the PRIMARY result; `delta` / `residual` use the
# granule-free residual layer, which is detection-dependent, and are the sensitivity arm.
from scipy.stats import mannwhitneyu

BASELINE_OF = {"delta_all": "all_extrasomatic", "residual_all": "all_extrasomatic",
               "delta": "residual_extrasomatic", "residual": "residual_extrasomatic"}

div_rows = []
for sample in C.SAMPLES:
    df = axis1[axis1["sample"] == sample]
    m = df["is_marker"]
    for stat in ["delta_all", "residual_all", "delta", "residual"]:
        u, pv = mannwhitneyu(df.loc[m, stat], df.loc[~m, stat], alternative="greater")
        # rank-biserial effect size -- reported alongside p because n = 290 genes makes small
        # differences significant on their own
        n1, n2 = int(m.sum()), int((~m).sum())
        div_rows.append(dict(sample=sample, statistic=stat,
                             baseline=BASELINE_OF[stat],
                             is_primary=BASELINE_OF[stat] == "all_extrasomatic",
                             n_marker=n1, n_other=n2,
                             median_marker=float(df.loc[m, stat].median()),
                             median_other=float(df.loc[~m, stat].median()),
                             u=float(u), pval=float(pv),
                             rank_biserial=float(2 * u / (n1 * n2) - 1)))

div = pd.DataFrame(div_rows)
div["star"] = div["pval"].apply(A3.p_val_to_star)
div.to_csv(OUT / "axis1_divergence_test.csv", index=False)
display(div)


,sample,statistic,baseline,is_primary,n_marker,n_other,median_marker,median_other,u,pval,rank_biserial,star
0,WT,delta_all,all_extrasomatic,True,20,270,0.228219,-0.601747,4773.0,5.105311e-09,0.767778,***
1,WT,residual_all,all_extrasomatic,True,20,270,1.134331,0.201252,4652.0,3.468050e-08,0.722963,***
2,WT,delta,residual_extrasomatic,False,20,270,0.250563,-0.645216,4773.0,5.105311e-09,0.767778,***
3,WT,residual,residual_extrasomatic,False,20,270,1.228583,0.211811,4671.0,2.585565e-08,0.730000,***
4,AD,delta_all,all_extrasomatic,True,20,270,0.082622,-0.881552,4568.0,1.230200e-07,0.691852,***
5,AD,residual_all,all_extrasomatic,True,20,270,1.083028,0.213317,4531.0,2.113452e-07,0.678148,***
6,AD,delta,residual_extrasomatic,False,20,270,0.089733,-0.935355,4568.0,1.230200e-07,0.691852,***
7,AD,residual,residual_extrasomatic,False,20,270,1.177144,0.235262,4541.0,1.827713e-07,0.681852,***


## 5. The non-seed, non-control gene test

**Section 4 cannot answer the reviewer's question, and we should say so before he does.** mcDETECT
defines a granule by running DBSCAN on the 20 marker genes and drawing the minimum enclosing sphere
around the resulting cluster. Marker transcripts are therefore concentrated inside granules *by
construction*. That the markers sit above a line fitted to the other genes is consistent with the
compartment being real, but it is not evidence for it: the same picture would appear if granules
were nothing but locally dense ambient RNA around marker transcripts. Fitting the line on
non-markers only (`a3_common.axis1_table`) makes 18/20 an out-of-sample statement; it does not
make it non-circular.

**So re-ask the question on genes that had no hand in defining a granule.** Two exclusions:

| excluded | why | n |
|---|---|---|
| `C.SYN_GENES` | seeded the DBSCAN pass that created every sphere | 20 |
| the published 19-gene NC list | `nc_filter` removed spheres enriched in them, so their depletion in Set 2 is circular too | 19 |

Gria2 is on both lists, so 38 unique genes come out and **252 of the 290 panel genes remain**.
Dropping the NC genes matters: `Gjc3` and `Opalin` are the two most granule-depleted genes in the
panel and both are negative controls, so keeping them would have manufactured part of the result.

**The labels are the panel's own design sheet.** `gene_panel.csv` -- the same file `select_set0`
reads -- carries three curated annotation columns written when the probe set was chosen, years
before any granule was called. Nothing about them is derived from mcDETECT's output. All three are
tested and all three are reported, including the one that does not separate: `Neuropil` mixes
subcellular localisation with panel provenance (`Xenium` is one of its values), and disclosing a
null costs less than being asked why one column of three was shown.

**Two statistics, built completely differently.** `logFC_granule_vs_residual` from section 4 is the
primary: granule against residual non-somatic RNA **within the same 50 um spot**, so "granules
simply sit in neuropil" cannot generate it. `residual_all` from section 3 is the robustness arm --
a whole-section compositional residual with no spatial matching at all. Each gene contributes one
value, so the pseudo-replication that makes the count model's own p-values anticonservative does
not touch this test.


In [7]:
# THE NON-CIRCULAR ARM. Everything above is marker-anchored; this is not.
#
# One Mann-Whitney per (statistic, contrast, sample), always one-sided in the direction the
# compartment hypothesis predicts, over genes that neither seeded detection nor entered nc_filter.
# Same rank-biserial convention as section 4 so the two tables can be read side by side.
if RUN_NONSEED:
    from scipy.stats import mannwhitneyu

    seeds = set(C.SYN_GENES)
    ncs = set(A3.load_nc_genes())                      # 19 -- the list Set 2 was actually built on
    excluded = seeds | ncs
    panel_genes = set(A3.load_genes("WT"))
    neutral_set = set(A3.neutral_genes())
    assert neutral_set == panel_genes - excluded

    annot = A3.load_panel().rename(columns=lambda c: str(c).strip().replace("\ufeff", ""))
    annot = annot.rename(columns={annot.columns[0]: "gene"})
    for key, col in C.PANEL_ANNOT_COLS.items():
        annot[key] = annot[col].astype(str).str.strip().replace({"nan": ""})
    annot = annot[["gene"] + list(C.PANEL_ANNOT_COLS)]

    # BOTH statistics are recomputed over the NEUTRAL POOL, not the full panel. E(g) comes from
    # the neutral-pool count model written in section 4. R(g) is rebuilt here, because it is not
    # invariant to the change of pool either: the line through the neutral genes has slope ~1.18,
    # not 1, so the constant log shift in the baseline and in granule enrichment does not cancel
    # (measured: up to 0.46 on a gene). Read E(g) from its CSV rather than from `qpn` in memory so
    # this section does not depend on RUN_COUNT_MODEL having been on in this kernel.
    _qpn = pd.read_csv(OUT / "axis1_count_model_neutral.csv")

    r_rows = []
    for sample in C.SAMPLES:
        q = (parts_panel[parts_panel["sample"] == sample]
             .set_index("gene").reindex(sorted(neutral_set)).fillna(0))
        soma = q["intrasomatic"].to_numpy(float)
        gran = q["granule"].to_numpy(float)
        resid = q["residual_extrasomatic"].to_numpy(float)
        # the reviewer's two quantities, on a shared somatic reference, within the neutral pool
        B = A3.composition_logfc(gran + resid, soma)
        G = A3.composition_logfc(gran, soma)
        slope, intercept = np.polyfit(B, G, 1)
        r_rows.append(pd.DataFrame(dict(sample=sample, gene=sorted(neutral_set),
                                        baseline_neutral=B, granule_enrichment_neutral=G,
                                        residual_all=G - (slope * B + intercept))))
        print(f"[{sample}] neutral-pool baseline line: slope {slope:.3f}, "
              f"intercept {intercept:.3f}")
    rtab = pd.concat(r_rows, ignore_index=True)

    ns = (_qpn[["sample", "gene", C.NONSEED_PRIMARY_STAT]]
          .merge(rtab, on=["sample", "gene"])
          .merge(annot, on="gene", how="left"))
    ns = ns[ns["gene"].isin(panel_genes) & ~ns["gene"].isin(excluded)].reset_index(drop=True)
    for key in C.PANEL_ANNOT_COLS:
        ns[key] = ns[key].fillna("")

    n_clean = ns["sample"].value_counts()
    assert set(n_clean) == {len(panel_genes) - len(excluded & panel_genes)}, \
        f"clean gene count differs between samples: {dict(n_clean)}"
    print(f"excluded {len(seeds)} seed + {len(ncs)} NC genes ({len(excluded)} unique); "
          f"{int(n_clean.iloc[0])} of {len(panel_genes)} panel genes remain")

    # contrast -> (annotation key, group A, group B). A is the side the hypothesis predicts higher.
    CONTRASTS = {
        "neuronal_vs_glial": ("cell_type", C.NONSEED_NEURONAL, C.NONSEED_GLIAL),
        "synaptic_vs_unannotated": ("synapse", C.NONSEED_SYNAPSE, [""]),
        "neuropil_vs_unannotated": ("neuropil", C.NONSEED_NEUROPIL, [""]),
    }

    rows = []
    for stat in C.NONSEED_STATS:
        for sample in C.SAMPLES:
            d = ns[ns["sample"] == sample]
            for name, (key, ga, gb) in CONTRASTS.items():
                a = d.loc[d[key].isin(ga), stat].dropna()
                b = d.loc[d[key].isin(gb), stat].dropna()
                u, pv = mannwhitneyu(a, b, alternative="greater")
                n1, n2 = len(a), len(b)
                rows.append(dict(sample=sample, statistic=stat, contrast=name,
                                 is_primary=(stat == C.NONSEED_PRIMARY_STAT
                                             and name == "neuronal_vs_glial"),
                                 n_a=n1, n_b=n2,
                                 median_a=float(a.median()), median_b=float(b.median()),
                                 # above zero = a larger share of granule RNA than of the RNA
                                 # around it. Interpretable for E(g); for R(g) the OLS residual
                                 # is centred at zero by construction, so it is descriptive only.
                                 n_a_above0=int((a > 0).sum()), n_b_above0=int((b > 0).sum()),
                                 pval=float(pv),
                                 rank_biserial=float(2 * u / (n1 * n2) - 1)))
    nsa = pd.DataFrame(rows)
    nsa["star"] = nsa["pval"].apply(A3.p_val_to_star)
    nsa.to_csv(OUT / "axis1_nonseed_annotation.csv", index=False)

    # Reproducibility across the two sections. WT and AD were detected independently, so a shared
    # ranking of genes that seeded nothing is not something a passive ambient sample would produce.
    rep = []
    for stat in C.NONSEED_STATS:
        w = ns[ns["sample"] == "WT"].set_index("gene")[stat]
        a = ns[ns["sample"] == "AD"].set_index("gene")[stat]
        ix = w.index.intersection(a.index)
        rho, pv = spearmanr(w[ix], a[ix])
        rep.append(dict(statistic=stat, n_genes=len(ix), spearman_rho=float(rho), pval=float(pv)))
    nsr = pd.DataFrame(rep)
    nsr.to_csv(OUT / "axis1_nonseed_reproducibility.csv", index=False)

    # The exclusion arithmetic, written out so the response document quotes it rather than
    # re-deriving it (20 + 19 - 1 overlap = 38 is not recoverable from the gene table alone).
    pd.DataFrame([dict(n_panel=len(panel_genes), n_seed=len(seeds), n_nc=len(ncs),
                       n_both=len(seeds & ncs), n_excluded=len(excluded & panel_genes),
                       n_clean=int(n_clean.iloc[0]),
                       both_genes="; ".join(sorted(seeds & ncs)))]
                 ).to_csv(OUT / "axis1_nonseed_scope.csv", index=False)

    ns.to_csv(OUT / "axis1_nonseed_genes.csv", index=False)
    display(nsa[nsa["statistic"] == C.NONSEED_PRIMARY_STAT])
    display(nsr)


[WT] neutral-pool baseline line: slope 1.052, intercept -0.024
[AD] neutral-pool baseline line: slope 0.986, intercept -0.207
excluded 20 seed + 19 NC genes (38 unique); 252 of 290 panel genes remain


,sample,statistic,contrast,is_primary,n_a,n_b,median_a,median_b,n_a_above0,n_b_above0,pval,rank_biserial,star
0,WT,logFC_granule_vs_residual,neuronal_vs_glial,True,17,39,0.411642,-1.028618,14,2,1.351227e-08,0.942685,***
1,WT,logFC_granule_vs_residual,synaptic_vs_unannotated,False,35,217,0.277406,0.114183,28,125,4.314029e-03,0.276893,**
2,WT,logFC_granule_vs_residual,neuropil_vs_unannotated,False,44,183,0.120720,0.114995,27,104,7.038888e-01,-0.051913,ns
3,AD,logFC_granule_vs_residual,neuronal_vs_glial,True,17,39,0.205691,-1.203620,9,1,2.760085e-07,0.849170,***
4,AD,logFC_granule_vs_residual,synaptic_vs_unannotated,False,35,217,0.045112,-0.091869,21,95,1.475409e-02,0.229493,*
5,AD,logFC_granule_vs_residual,neuropil_vs_unannotated,False,44,183,0.160905,-0.130849,28,73,6.522507e-04,0.312469,***


,statistic,n_genes,spearman_rho,pval
0,logFC_granule_vs_residual,252,0.761827,4.747446e-49
1,residual_all,252,0.746767,3.526810e-46


## 6. Axis 2 -- the condition contrast across compartments

The reviewer's framing sentence names "differences between **regions or conditions**", so the
compartment axis alone does not close the point.

**The subdomain arm already exists and is not recomputed.**
`output/MERSCOPE_WT_AD_comparison/neuropil_subdomains_Isocortex_50/` holds
`{granule,cell,ambient}_DE_genes_Subdomain 1_vs_Subdomain 2.csv`, and A1 already reports the
granule layer against those floors (rho = 0.37 / 0.42). What is **missing** is the WT-vs-AD
contrast on the same three layers over the same grid -- that is the one that maps onto the ambient
bias concern, and it is computed below.

**Two reporting rules, both load-bearing:**

1. **Significant-gene counts are not comparable across layers.** The layers differ enormously in
   counts per spot and in sparsity, and a rank test's power tracks that -- which is why the
   ambient and cell layers show 253 and 234 significant genes against the granule layer's 161.
   Compare **rankings and logFC correlations only**. Quoting the tallies invites the reading *"the
   authors' ambient layer yields more DE genes than their granule layer."*
2. **n = 1 vs 1.** One WT section, one AD section, so every spot-level WT/AD p-value is
   pseudo-replication -- and this applies to the published result too. WT/AD is reported
   descriptively; the inferential weight sits on §4's within-sample divergence.

In [8]:
if RUN_AXIS2:
    # WT-vs-AD logFC per layer, from the transcript-level partition. NOT detection-independent:
    # `residual_extrasomatic` is "extrasomatic AND not inside a called sphere", so like Axis 1's
    # sensitivity arm it is defined by the granule calls. What it IS free of is the published
    # ambient layer's construction errors -- assignment is per transcript, so there is no
    # spot-boundary misallocation, no double counting across overlapping spheres and no
    # max(..., 0) clip. Axis 1 is where the detection-independent baseline lives.
    wide = parts_panel.pivot_table(index="gene", columns="sample", values=C.DE_LAYERS)
    a2_rows = []
    for layer in C.DE_LAYERS:
        wt = wide[(layer, "WT")].fillna(0).to_numpy(float)
        ad = wide[(layer, "AD")].fillna(0).to_numpy(float)
        lfc = A3.composition_logfc(ad, wt)
        a2_rows.append(pd.DataFrame({"gene": wide.index, "layer": layer, "logFC_AD_vs_WT": lfc}))
    a2 = pd.concat(a2_rows, ignore_index=True)
    a2["is_marker"] = a2["gene"].isin(C.SYN_GENES)
    a2.to_csv(OUT / "axis2_wt_ad_by_layer.csv", index=False)

    piv = a2.pivot_table(index="gene", columns="layer", values="logFC_AD_vs_WT")
    corr_rows = []
    for a in C.DE_LAYERS:
        for b in C.DE_LAYERS:
            if a >= b:
                continue
            rho, pv = spearmanr(piv[a], piv[b])
            corr_rows.append(dict(layer_a=a, layer_b=b, spearman_rho=float(rho),
                                  pval=float(pv), n_genes=int(len(piv))))
    corr = pd.DataFrame(corr_rows)
    corr.to_csv(OUT / "axis2_layer_correlation.csv", index=False)
    print("WT/AD logFC agreement between layers (rankings only -- NOT gene counts):")
    display(corr)

    # Pointer to the already-computed subdomain contrast; deliberately not recomputed.
    pub = sorted(C.PUBLISHED_SUBDOMAIN_DIR.glob("*_DE_genes_*.csv"))
    print(f"\npublished subdomain DE tables ({len(pub)} found, NOT recomputed):")
    for f in pub:
        print("  ", f.name)

WT/AD logFC agreement between layers (rankings only -- NOT gene counts):


,layer_a,layer_b,spearman_rho,pval,n_genes
0,intrasomatic,residual_extrasomatic,0.632286,8.808691e-34,290
1,granule,intrasomatic,0.614070,1.902151e-31,290
2,granule,residual_extrasomatic,0.787629,1.600985e-62,290



published subdomain DE tables (5 found, NOT recomputed):
   ambient_DE_genes_Subdomain 1_vs_Subdomain 2.csv
   cell_DE_genes_Subdomain 1_vs_Subdomain 2.csv
   cell_DE_genes_Subdomain 1_vs_Subdomain 2_GSEA.csv
   granule_DE_genes_Subdomain 1_vs_Subdomain 2.csv
   granule_DE_genes_Subdomain 1_vs_Subdomain 2_GSEA.csv


In [9]:
if RUN_AXIS2:
    # Region stratification, on the same partition. Uses the 25um WHOLE-SECTION grid: the 50um
    # object covers only Isocortex + FT (4,310 spots), which cannot carry a region contrast.
    spots25 = sc.read_h5ad(C.SUBDOMAIN_SPOTS_25)
    print("25um grid:", spots25.shape, "| areas:",
          spots25.obs["brain_area"].value_counts().to_dict())
    print("\nNOTE:", C.SOMATIC_LAYER_NOTE)
    print("\nlayers present:", list(spots25.layers))

25um grid: (121084, 290) | areas: {'Unknown': 45124, 'Isocortex': 18692, 'MB': 16248, 'FT': 15452, 'HPF-SR': 7040, 'HPF-CA': 6964, 'OLF': 3696, 'TH': 3312, 'HPF-DG': 3296, 'CTXsp': 1260}

NOTE: Do NOT derive the somatic layer as X - extrasomatic: the two were filled by different assignment rules and X >= extrasomatic fails in 3.07% of cells of the 50um object (worst deficit 46 transcripts). Rebuild it with fill_spot_expression on transcripts[overlaps_nucleus == 1], exactly as code/5_neuropil_subdomains_data.py:242-243 does, so somatic and non-somatic share one counting convention.

layers present: ['extrasomatic_transcripts']


## 7. Correctness gates

On by default.

In [10]:
if VALIDATE:
    # (a) the partition must be exact, per gene per sample -- if the three arms do not sum to the
    #     transcript count, they are not a partition and every logFC above is on a wrong
    #     denominator.
    for sample in C.SAMPLES:
        p = parts[parts["sample"] == sample]
        assert (p[C.DE_LAYERS].sum(axis=1) == p["n_total"]).all(), \
            f"{sample}: layers do not sum to n_total"
        tx_n = len(A3.load_transcripts(sample, columns=["target"], verbose=False))
        assert int(p["n_total"].sum()) == tx_n, \
            f"{sample}: partition total {int(p['n_total'].sum()):,} != {tx_n:,}"
        print(f"[ok] partition exact for {sample}: {tx_n:,} transcripts")

    # (b) the arms are disjoint BY CONSTRUCTION (a single np.where cascade with soma first), so
    #     re-deriving the partition to assert it would just re-run the most expensive step in A3
    #     to check an identity. Assert on the cached labels instead.
    for sample in C.SAMPLES:
        cached = C.transcript_layer_path(sample)
        if not cached.exists():
            continue
        codes = pd.read_parquet(cached)["layer"].to_numpy()
        assert set(np.unique(codes)) <= {0, 1, 2}, "unexpected layer code"
        print(f"[ok] {sample}: cached layers are a clean 3-way partition "
              f"({np.bincount(codes, minlength=3)})")

    # (c) the sensitivity baseline must be granule-free. If any in-granule transcript leaked into
    #     residual_extrasomatic the two baselines would stop being distinct arms. Granule-free is
    #     NOT the same as detection-independent: residual_extrasomatic is defined by the calls, so
    #     it is detection-DEPENDENT. The detection-independent baseline is the PRIMARY one,
    #     granule + residual_extrasomatic, which needs no sphere at all.
    print("[ok] residual_extrasomatic is granule-free by construction "
          "(detection-DEPENDENT; the primary baseline is granule + residual)")

    # (e) the soma reference must cancel out of `delta`. Section 3's prose claims
    #     delta == log2 share(granule) - log2 share(residual_extrasomatic) exactly; if that fails,
    #     the two logFCs are not on a shared denominator and the whole axis is mis-stated.
    for sample in C.SAMPLES:
        df = axis1[axis1["sample"] == sample]
        direct = A3.composition_logfc(df["n_granule"].to_numpy(),
                                      df["n_residual_extrasomatic"].to_numpy())
        assert np.abs(df["delta"].to_numpy() - direct).max() < 1e-9, \
            f"{sample}: the soma reference does not cancel out of delta"
        direct_all = A3.composition_logfc(df["n_granule"].to_numpy(),
                                          df["n_all_extrasomatic"].to_numpy())
        assert np.abs(df["delta_all"].to_numpy() - direct_all).max() < 1e-9, \
            f"{sample}: the soma reference does not cancel out of delta_all"
    print("[ok] the soma reference cancels exactly out of delta and delta_all")

    # (d) composition_logfc must reproduce benchmark_diffusion.ipynb's baseline on the same input
    a = np.array([10.0, 20.0, 30.0])
    b = np.array([30.0, 20.0, 10.0])
    eps = C.AXIS1_PSEUDOCOUNT
    ref = np.log2((a + eps) / (a.sum() + eps)) - np.log2((b + eps) / (b.sum() + eps))
    assert np.allclose(A3.composition_logfc(a, b), ref)
    print("[ok] composition_logfc matches the published form")
    # (f) the non-seed test must contain NO seed gene and NO negative-control gene. This is the
    #     single assumption section 5 rests on -- if it fails, the section is circular after all
    #     and must not be reported.
    if RUN_NONSEED:
        leaked = set(ns["gene"]) & (set(C.SYN_GENES) | set(A3.load_nc_genes()))
        assert not leaked, f"seed/NC genes leaked into the non-seed test: {sorted(leaked)}"
        assert ns["sample"].nunique() == len(C.SAMPLES)
        print(f"[ok] non-seed test is clean: {ns['gene'].nunique()} genes, "
              f"no seed and no negative control")


[ok] partition exact for WT: 103,398,068 transcripts


[ok] partition exact for AD: 68,876,647 transcripts


[ok] WT: cached layers are a clean 3-way partition ([28838683  6128938 68430447])


[ok] AD: cached layers are a clean 3-way partition ([20901924  3677422 44297301])
[ok] residual_extrasomatic is granule-free by construction (detection-DEPENDENT; the primary baseline is granule + residual)
[ok] the soma reference cancels exactly out of delta and delta_all
[ok] composition_logfc matches the published form
[ok] non-seed test is clean: 252 genes, no seed and no negative control


## Outputs

| file | contents |
|---|---|
| `partition_counts.csv` | per gene per sample, transcripts in each of the three disjoint arms |
| `transcript_layer_<sample>.parquet` | cached per-transcript layer label; used only within this notebook |
| `spot_layer_counts.parquet` | per (spot, gene, layer) counts -- the estimable count model's input |
| `clip_bias_by_gene.csv` | fraction of spots where the published `extrasomatic - granule` subtraction goes negative before clipping, marker vs non-marker |
| `clip_bias_scope.csv` | what that number was measured on: the published 50 um Isocortex + FT ROI, its spot and granule counts, and the granule assignment rate |
| `axis1_gene_table.csv` | `baseline_logFC`, `granule_enrichment`, `delta`, the non-marker regression line and each gene's residual |
| `axis1_summary.csv` | Spearman rho, regression coefficients, markers above the diagonal and above the line |
| `axis1_count_model_neutral.csv` | the same count model over the 252-gene neutral pool, so logFC = 0 is the neutral point | 4 |
| `axis1_nonseed_annotation.csv` | the non-circular test: annotation contrasts over the 252 genes that seeded nothing and filtered nothing | 5 |
| `axis1_nonseed_scope.csv` | the exclusion arithmetic: panel, seed, NC, overlap, clean gene counts | 5 |
| `axis1_nonseed_genes.csv` | per gene per sample: both statistics plus the three panel annotations, non-seed genes only | 5 |
| `axis1_nonseed_reproducibility.csv` | WT-vs-AD Spearman of each statistic over the same genes | 5 |
| `axis1_count_model.csv` | quasi-Poisson granule-vs-residual logFC per gene, with BH FDR -- the non-compositional primary |
| `axis1_divergence_test.csv` | one-sided marker-set enrichment on `delta` and on the residual, with rank-biserial effect size |
| `axis2_wt_ad_by_layer.csv` | WT-vs-AD logFC per gene in each of the three layers |
| `axis2_layer_correlation.csv` | rank agreement between layers -- **rankings only, never gene counts** |